# max_peaks=100, then ranked by chromatin_od -- precision@K (K = 10, 20, 30) and re-ranking overhead

**What this notebook is.** `max_peaks_100_variant.ipynb` (this same directory) measured how much
faster the pipeline gets when `max_peaks=100` caps the pre-NMS candidate pool, ranking the
survivors by `tm_score`. This notebook keeps that same `max_peaks=100` cap fixed in **both**
branches -- it is no longer the thing under test -- and ablates the **ranking criterion** applied
to the resulting ~90-99-candidate post-NMS pool instead: `baseline` ranks it by `tm_score`
(`TM_CCOEFF`, D5's production ranker, free -- the score matchTemplate already produced);
`variant` ranks the *same* pool by `chromatin_od` (`od51` --
`midog_utils.chromatin.chromatin_density`, window=51, darkest-10%, one of D5's "what would
change my mind" axes).

**Why this differs from the ranking-only version of this ablation.** Ranking the *unbounded*
pool by `chromatin_od` means computing `chromatin_density` for the 15,000-19,000 candidates the
search produces before NMS thins them -- `latency_profiling/chromatin_od_latency_profile.ipynb`
measured that at ~645ms mean per ROI, heavy enough to need thermal-throttling protection on this
machine. Applying `max_peaks=100` **first** collapses that loop to the ~90-99 candidates that
survive the cap and NMS, which is the actual question this task asks: not "what does
`chromatin_od` cost on every candidate the search finds," but "what does it cost once we've
already paid the `max_peaks=100` win for speed."

**Branch structure.** The ranking criterion cannot change which candidates survive extraction and
NMS -- only the order they come out in -- so stages 1-5 (search through NMS, `max_peaks=100`
applied once) run **once** per ROI, and only stage 6 forks into `t6_baseline` (sort by `score`)
and `t6a`/`t6b`/`t6c` (pad, compute `chromatin_density` per survivor, sort by it).

**Everything else is held fixed at production's accepted defaults** (`D8_TEMPLATE_ANCHOR.md`,
current as of 2026-09-10): `hematoxylin_od` channel, `TM_CCOEFF` (unnormalized), single-scale/
single-angle/no-flip augmentation, `peak_min_distance=7`, `self_hit_radius=5.0`, NMS radius =
match radius = 7.5 um, the same 14 ROIs, one seed per ROI, `tightened_template_box` seed
refinement, one `matchTemplate` pass per ROI shared by both branches.

**Oracle.** `max_peaks_100_variant.ipynb`'s own committed output
(`max_peaks_100_timing.csv`, `max_peaks_100_precision.csv`) already reports exactly this
notebook's `baseline` branch -- `max_peaks=100`, ranked by `tm_score` -- as its own `variant`
branch (that notebook ablated the cap at fixed `tm_score` ranking; this one fixes the cap and
ablates the ranking). Verification 1 below checks against it. There is no pre-existing oracle for
`max_peaks=100` ranked by `chromatin_od` -- no notebook in this repo has run that combination
before -- so Verification 2 is a sanity check on `od51` itself, not a reproduction.

**Caveat (D5, restated -- same as the template).** Single-seed, n=14: one click per ROI, not the
paired, multi-seed sweep D5 sets as the bar for changing a production default. `midog_utils/chromatin.py`'s
own module docstring: no experiment may declare a chromatin axis primary without measuring, on
its own data, that it beats `tm_score`.

**Sequential execution matters for the timing half of this question.** Do not execute this
notebook concurrently with either sibling ablation notebook in this directory, or with any
`latency_profiling/*.ipynb`.

In [1]:
import gc
import sys
import time

import cv2
import numpy as np
import pandas as pd

sys.path.insert(0, '..')
from midog_utils import channels as ch
from midog_utils import chromatin as cm
from midog_utils import dataset as ds
from midog_utils import evaluate as ev
from midog_utils import find_and_suppress as fs
from midog_utils import seed_selection as ss
from midog_utils import template_match as tm
from midog_utils import compare as cp
from midog_utils.nms import nms_by_distance

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 220)

NB_T0 = time.time()

# ---------------------------------------------------------------------------------------
# Config -- identical to max_peaks_100_variant.ipynb's cell 1, except MAX_PEAKS is now fixed
# at 100 (that notebook's "variant" value) in *both* branches: this notebook ablates the
# ranking criterion, not the candidate cap, and the cap is no longer the thing under test.
# ---------------------------------------------------------------------------------------
IMAGES_DIR = '../images/extra_valid'
SEED_INDEX = 0

CHANNEL = 'hematoxylin_od'
METHOD = cv2.TM_CCOEFF
PEAK_MIN_DISTANCE = 7
SELF_HIT_RADIUS = 5.0
DEEP_FLOOR_Z = -1.5           # unchanged -- same permissive extraction floor as the template
MAX_PEAKS = 100                # max_peaks_100_variant.ipynb's "variant" value -- fixed here in
                                # both branches, not ablated

NMS_RADIUS_UM = ev.MIDOG_RADIUS_UM
MATCH_RADIUS_UM = ev.MIDOG_RADIUS_UM

CFG = fs.FSConfig(channel=CHANNEL, base_size=tm.BASE_SIZE, scales=(1.0,),
                  n_angles=1, flips=(False,), peak_min_distance=PEAK_MIN_DISTANCE,
                  self_hit_radius=SELF_HIT_RADIUS)
BORDER = CFG.patch_size // 2
OTSU_WINDOW = tm.BASE_SIZE

# The ablation: two arms rank the *same* max_peaks=100-capped, post-NMS pool. 'tm_score' sorts
# on the score matchTemplate already produced (free -- no new computation). 'chromatin_od' needs
# a new per-candidate feature, cm.chromatin_density(window=51) == production's 'od51' axis (the
# chromatin sweep notebooks' AXES['chromatin_od'] = 'od51'), computed on far fewer candidates
# than the unbounded pool because max_peaks=100 has already thinned the field.
OD51_WINDOW = 51                # tm.BASE_SIZE -- matches the accepted chromatin sweep's od51
OD_PAD_51 = OD51_WINDOW // 2    # 25 -- exactly covers a 51px window; chromatin_od_latency_profile.ipynb
                                 # smoke-tested this against the 60px pad sized for od_ctx: identical od51
VARIANT_LABEL = 'chromatin_od (od51) ranking, max_peaks=100'

BUDGETS_FOR_EVAL = (10, 20, 30)   # same restricted scope as max_peaks_100_variant.ipynb
BUDGET = max(BUDGETS_FOR_EVAL)    # 30 -- rank-and-truncate target for both arms; comfortably
                                   # below the ~90-99 candidates max_peaks=100 typically leaves

MAX_PEAKS_ORACLE_TIMING_CSV = 'max_peaks_100_timing.csv'
MAX_PEAKS_ORACLE_PRECISION_CSV = 'max_peaks_100_precision.csv'

print(f'BUDGETS_FOR_EVAL={BUDGETS_FOR_EVAL}, NMS radius = match radius = {NMS_RADIUS_UM} um, '
      f'channel={CHANNEL}, MAX_PEAKS={MAX_PEAKS} (fixed, both branches)')
print(f'chromatin_od (od51): window={OD51_WINDOW} frac={cm.DEFAULT_FRAC} OD_PAD_51={OD_PAD_51}')

BUDGETS_FOR_EVAL=(10, 20, 30), NMS radius = match radius = 7.5 um, channel=hematoxylin_od, MAX_PEAKS=100 (fixed, both branches)
chromatin_od (od51): window=51 frac=0.1 OD_PAD_51=25


## Helpers

In [2]:
def draw_seed_with_retry(pool, rng, check_fn):
    """Draw a row via `rng.integers`; on failure drop it and redraw on the same stream."""
    working, retries = pool.copy(), 0
    while len(working) > 0:
        idx = int(rng.integers(len(working)))
        row = working.iloc[idx]
        result = check_fn(row)
        if result is not None:
            return row, result, retries
        working = working.drop(working.index[idx])
        retries += 1
    raise ValueError('seed pool exhausted -- no candidate passed check_fn')


def suppress(centers, scores, radius, ref_xy):
    """NMS at `radius`, then drop the template's own self-correlation."""
    keep = nms_by_distance(centers, scores, radius)
    c, s = centers[keep], scores[keep]
    if len(c):
        ok = np.hypot(c[:, 0] - ref_xy[0], c[:, 1] - ref_xy[1]) > SELF_HIT_RADIUS
        c, s = c[ok], s[ok]
    return c, s


def roi_files(images_dir=IMAGES_DIR):
    import os
    return sorted(f for f in os.listdir(images_dir) if f.endswith('.tiff'))


print(f'{len(roi_files())} ROIs on disk in {IMAGES_DIR}/')

14 ROIs on disk in ../images/extra_valid/


## Per-ROI worker

In [3]:
def run_roi(fn, image_id, domain, anns):
    gc.disable()

    # =================== SETUP (once per ROI; not part of click latency) ==============
    t0 = time.perf_counter()
    path = f'{IMAGES_DIR}/{fn}'
    rgb = ds.load_roi(path)
    mpp = ds.roi_mpp(path)
    roi_shape = rgb.shape
    match_radius = ev.radius_px(mpp, MATCH_RADIUS_UM)
    nms_radius = ev.radius_px(mpp, NMS_RADIUS_UM)
    hem = ch.to_channel(rgb, CHANNEL)
    gray_inv = ch.to_gray_inverted(rgb)
    H, W = hem.shape[:2]
    del rgb

    gt = ds.image_annotations(anns, fn)
    seed_pool, flagged = ss.agreement_pool(gt[gt['category_id'] == ds.MITOTIC])
    seed_pool = ss.border_filter(seed_pool, BORDER, roi_shape)
    rng = np.random.default_rng([SEED_INDEX, image_id])

    def _check(row):
        r = ss.tightened_template_box(gray_inv, float(row['cx']), float(row['cy']),
                                      otsu_window=OTSU_WINDOW)
        if r is None:
            return None
        if tm.read_padded_patch(hem, r[1], r[2], CFG.patch_size) is None:
            return None
        return r

    seed, seed_tpl_probe, n_retries = draw_seed_with_retry(seed_pool, rng, _check)
    seed_ann_id = int(seed['ann_id'])
    click_cx, click_cy = float(seed['cx']), float(seed['cy'])
    gt_eval = gt[gt['ann_id'] != seed_ann_id].reset_index(drop=True)
    n_gt = int((gt_eval['category_id'] == ds.MITOTIC).sum())
    t_setup = time.perf_counter() - t0

    # ============ PIPELINE stages 1-5 (shared by both branches; run once) ==============
    # The ranking criterion cannot change which candidates survive extraction and NMS --
    # only the order they come out in -- so there is one pool, ranked two ways, not two
    # pools. max_peaks=100 is applied once, here, identically for both branches.
    stages = {}
    t_outer0 = time.perf_counter()

    t1 = time.perf_counter()
    r = ss.tightened_template_box(gray_inv, click_cx, click_cy, otsu_window=OTSU_WINDOW)
    assert r is not None, f'{fn}: seed was pre-validated in SETUP but refused here'
    border_probe = tm.read_padded_patch(hem, r[1], r[2], CFG.patch_size)
    assert border_probe is not None, f'{fn}: seed was pre-validated in SETUP but unreadable here'
    assert r == seed_tpl_probe, f'{fn}: single-shot refinement diverged from the SETUP search'
    base_size, tpl_cx, tpl_cy = r
    tpl_xy = (float(tpl_cx), float(tpl_cy))
    stages['t1_refine_seed_box_s'] = time.perf_counter() - t1

    t2 = time.perf_counter()
    patch = tm.read_padded_patch(hem, *tpl_xy, CFG.patch_size)
    templates, _ = tm.build_augmentations(patch, base_size, CFG.scales, CFG.n_angles, CFG.flips)
    stages['t2_patch_template_build_s'] = time.perf_counter() - t2

    t3 = time.perf_counter()
    PAD = max((t.shape[0] - 1) // 2 for t in templates)
    hem_p = cv2.copyMakeBorder(hem, PAD, PAD, PAD, PAD, borderType=cv2.BORDER_REPLICATE)
    fused_p, _, valid_p = tm.fused_response(hem_p, templates, CFG.scale_normalize, method=METHOD)
    stages['t3_template_matching_s'] = time.perf_counter() - t3

    fused = fused_p[PAD:PAD + H, PAD:PAD + W]
    valid = valid_p[PAD:PAD + H, PAD:PAD + W]
    assert bool(valid.all()), f'{fn}: padding left part of the ROI unreachable (PAD={PAD})'
    med, mad = tm.robust_stats(fused, valid)

    t4 = time.perf_counter()
    cut = med + DEEP_FLOOR_Z * mad
    centers, scores = tm.extract_peaks(fused, valid, PEAK_MIN_DISTANCE, cut, MAX_PEAKS)
    stages['t4_threshold_peak_extraction_s'] = time.perf_counter() - t4
    n_peaks = len(centers)
    # No "MAX_PEAKS never binds" assertion here, unlike the unbounded-pool config -- at
    # MAX_PEAKS=100 the cap is *meant* to bind; that is the whole premise of this notebook.

    t5 = time.perf_counter()
    c, s = suppress(centers, scores, nms_radius, tpl_xy)
    stages['t5_nms_selfhit_s'] = time.perf_counter() - t5
    assert bool(np.all(np.diff(s) <= 0)), f'{fn}: post-NMS pool is not score-descending'

    pool = pd.DataFrame({'cx': c[:, 0], 'cy': c[:, 1], 'score': s})
    n_detections = len(pool)

    d_seed = (np.hypot(pool['cx'] - tpl_xy[0], pool['cy'] - tpl_xy[1]) if len(pool) else np.array([]))
    n_near_seed = int((d_seed <= match_radius).sum())

    # ===================== Branch: baseline -- rank by tm_score (free) ===================
    # `sort_values` returns a new frame; `pool` itself is never reordered, so the NMS
    # score-descending order `evaluate_arms`'s stable mergesort relies on for tie-breaking
    # is preserved for the chromatin_od branch below.
    t6 = time.perf_counter()
    top_tm = pool.sort_values('score', ascending=False, na_position='last',
                              kind='mergesort').head(BUDGET)
    stages['t6_baseline_s'] = time.perf_counter() - t6

    # ================ Branch: variant -- rank by chromatin_od (od51) ====================
    # od51 must be computed for the *entire* post-NMS pool, not just the eventual top-K --
    # which candidates land in the top-30 by od51 is exactly what is unknown before it is
    # computed. n_detections here runs ~90-99 (max_peaks=100 already applied above), not the
    # 15,000-19,000 the unbounded pipeline leaves -- this is the added step at the scale the
    # task actually asked about.
    t6a = time.perf_counter()
    hem_pad = cv2.copyMakeBorder(hem, OD_PAD_51, OD_PAD_51, OD_PAD_51, OD_PAD_51, cv2.BORDER_REPLICATE)
    px = pool['cx'].to_numpy() + OD_PAD_51
    py = pool['cy'].to_numpy() + OD_PAD_51
    stages['t6a_od_pad_s'] = time.perf_counter() - t6a

    t6b = time.perf_counter()
    pool['od51'] = [cm.chromatin_density(hem_pad, x, y, window=OD51_WINDOW) for x, y in zip(px, py)]
    stages['t6b_od51_loop_s'] = time.perf_counter() - t6b

    t6c = time.perf_counter()
    top_od51 = pool.sort_values('od51', ascending=False, na_position='last',
                                kind='mergesort').head(BUDGET)
    stages['t6c_variant_rank_s'] = time.perf_counter() - t6c

    t_outer_total = time.perf_counter() - t_outer0
    gc.enable()

    od51_tie, od51_nan_rate = cp._tie_and_nan(pool['od51'])
    assert od51_nan_rate == 0.0, f'{fn}: {od51_nan_rate} od51 NaN rate -- OD_PAD_51 too small'

    # ============== Both arms, one pool, independent rank + re-match per arm =============
    tm_arm = cp.Arm('tm_score', (lambda d=pool: d), rank_key='score', seeded=True,
                    z=DEEP_FLOOR_Z, z_dependent=True, nms_radius=nms_radius,
                    caps=(MAX_PEAKS,), coverage_key=fn)
    chromatin_arm = cp.Arm('chromatin_od', (lambda d=pool: d), rank_key='od51', seeded=True,
                           z=DEEP_FLOOR_Z, z_dependent=True, nms_radius=nms_radius,
                           caps=(MAX_PEAKS,), coverage_key=fn)
    ctx = dict(file_name=fn, tumor_type=domain, image_id=image_id, seed_ann_id=seed_ann_id,
              base_size=base_size, n_retries=n_retries,
              map_median=round(float(med), 5), mad_scale=round(float(mad), 5))
    checks = []
    ev_out = cp.evaluate_arms([tm_arm, chromatin_arm], gt_eval, match_radius, roi_shape=(H, W),
                              mpp=mpp, budgets=BUDGETS_FOR_EVAL, context=ctx, checks=checks)
    checks.append(dict(check='seed_annulus_empty', label=fn, n_near_seed=n_near_seed,
                       passed=bool(n_near_seed == 0)))

    del hem, gray_inv, hem_p, fused_p, valid_p, fused, valid, templates, patch, hem_pad
    del centers, scores, c, s
    gc.collect()

    t_shared = sum(stages[k] for k in ['t1_refine_seed_box_s', 't2_patch_template_build_s',
                                       't3_template_matching_s', 't4_threshold_peak_extraction_s',
                                       't5_nms_selfhit_s'])
    t_chromatin_od_overhead = (stages['t6a_od_pad_s'] + stages['t6b_od51_loop_s']
                               + stages['t6c_variant_rank_s'])
    t_baseline_pipeline = t_shared + stages['t6_baseline_s']
    t_variant_pipeline = t_shared + t_chromatin_od_overhead

    timing_row = dict(
        file_name=fn, tumor_type=domain, base_size=base_size, n_retries=n_retries,
        seed_ann_id=seed_ann_id, map_median=round(float(med), 5), mad_scale=round(float(mad), 5),
        n_peaks=n_peaks, n_detections=n_detections, roi_pixels=int(H) * int(W),
        t_setup_s=round(t_setup, 5),
        **{k: round(v, 5) for k, v in stages.items()},
        t_shared_stages_s=round(t_shared, 5),
        t_chromatin_od_overhead_s=round(t_chromatin_od_overhead, 5),
        t_baseline_pipeline_s=round(t_baseline_pipeline, 5),
        t_variant_pipeline_s=round(t_variant_pipeline, 5),
        t_outer_total_s=round(t_outer_total, 5),
        chromatin_od_nan_rate=od51_nan_rate, chromatin_od_largest_tie_block=od51_tie,
    )

    print(f"[{fn}] {domain:32s} base={base_size:2d} n_det={n_detections:4d} "
          f"baseline={t_baseline_pipeline*1000:7.2f}ms variant={t_variant_pipeline*1000:7.2f}ms "
          f"od_overhead={t_chromatin_od_overhead*1000:6.2f}ms "
          f"(+{100*t_chromatin_od_overhead/t_baseline_pipeline:.1f}%)", flush=True)

    return timing_row, ev_out, checks

## Run -- all 14 ROIs

In [4]:
images, annotations = ds.load_annotations('../databases/MIDOG++.json')
ds.check_invariants(annotations)
meta_ix = images.set_index('file_name')[['image_id', 'tumor_type']]

files = roi_files()
assert len(files) == 14, f'expected 14 ROIs in {IMAGES_DIR}/, found {len(files)}'

timing_rows, all_evs, all_checks = [], [], []
t_run = time.time()
for fn in files:
    image_id = int(meta_ix.loc[fn, 'image_id'])
    domain = meta_ix.loc[fn, 'tumor_type']
    trow, ev_out, checks = run_roi(fn, image_id, domain, annotations)
    timing_rows.append(trow)
    all_evs.append(ev_out)
    all_checks.extend(checks)
    gc.collect()

TIMING = pd.DataFrame(timing_rows).set_index('file_name')
ALL_RAW = pd.concat(all_evs, ignore_index=True)
CHECKS = pd.DataFrame(all_checks)

print(f'\n{len(files)} ROIs timed in {time.time() - t_run:.0f}s')
print(f'checks passed: {int(CHECKS["passed"].sum())}/{len(CHECKS)}')
assert CHECKS['passed'].all(), 'a pipeline-invariant check failed -- see CHECKS above'
CHECKS['check'].value_counts()

[013.tiff] human breast cancer              base=31 n_det=  97 baseline=1806.17ms variant=1916.55ms od_overhead=110.69ms (+6.1%)


[094.tiff] human breast cancer              base=25 n_det=  96 baseline=1338.66ms variant=1376.11ms od_overhead= 37.69ms (+2.8%)


[201.tiff] canine lung cancer               base=51 n_det=  98 baseline=1469.54ms variant=1510.37ms od_overhead= 41.06ms (+2.8%)


[233.tiff] canine lung cancer               base=25 n_det=  97 baseline=1187.29ms variant=1219.40ms od_overhead= 32.32ms (+2.7%)


[245.tiff] canine lymphosarcoma             base=47 n_det=  89 baseline=1345.93ms variant=1377.29ms od_overhead= 31.57ms (+2.3%)


...

[460.tiff] canine soft tissue sarcoma       base=47 n_det=  99 baseline=1302.29ms variant=1332.91ms od_overhead= 30.83ms (+2.4%)


[529.tiff] human melanoma                   base=37 n_det=  97 baseline=1396.65ms variant=1434.00ms od_overhead= 37.55ms (+2.7%)


[548.tiff] human melanoma                   base=29 n_det=  98 baseline=1402.31ms variant=1439.68ms od_overhead= 37.57ms (+2.7%)



14 ROIs timed in 66s
checks passed: 70/70


check
no_cap                28
nms_radius            28
seed_annulus_empty    14
Name: count, dtype: int64

## Verification 1 -- baseline (tm_score, max_peaks=100) branch reproduces `max_peaks_100_variant.ipynb`

That notebook's own `variant` branch (`max_peaks=100`, ranked by `tm_score`) is exactly this notebook's `baseline` branch -- the same cap, the same ranker, the same 14 ROIs and seed. If this fails, nothing below can be trusted.

In [5]:
oracle_timing = pd.read_csv(MAX_PEAKS_ORACLE_TIMING_CSV).set_index('file_name')
oracle_precision = pd.read_csv(MAX_PEAKS_ORACLE_PRECISION_CSV)
oracle_precision_variant = oracle_precision[oracle_precision['branch'] == 'variant'].copy()

oracle_roi = (oracle_timing[['seed_ann_id', 'base_size', 'n_detections_variant',
                             'map_median', 'mad_scale']]
             .rename(columns={'n_detections_variant': 'n_detections'}))
this_roi = TIMING[['seed_ann_id', 'base_size', 'n_detections', 'map_median', 'mad_scale']]

cmp = this_roi.join(oracle_roi, lsuffix='_this', rsuffix='_oracle')
mismatches = []
for col in ['seed_ann_id', 'base_size', 'n_detections']:
    bad = cmp[f'{col}_this'].astype(int) != cmp[f'{col}_oracle'].astype(int)
    if bad.any():
        mismatches.append((col, cmp.index[bad].tolist()))
for col in ['map_median', 'mad_scale']:
    bad = ~np.isclose(cmp[f'{col}_this'], cmp[f'{col}_oracle'], rtol=0, atol=1e-5)
    if bad.any():
        mismatches.append((col, cmp.index[bad].tolist()))

oracle_tp = oracle_precision_variant.pivot_table(index='file_name', values='tp_at_budget', columns='budget')
this_tm_tp = (ALL_RAW[ALL_RAW['arm'] == 'tm_score']
             .pivot_table(index='file_name', values='tp_at_budget', columns='budget'))
for k in BUDGETS_FOR_EVAL:
    this_k = this_tm_tp[k]
    oracle_k = oracle_tp.loc[this_k.index, k]
    bad = this_k.astype(int) != oracle_k.astype(int)
    if bad.any():
        mismatches.append((f'tp_at_{k}', this_k.index[bad].tolist()))

if mismatches:
    print('!! MISMATCH vs. max_peaks_100_variant.ipynb -- tm_score (max_peaks=100) branch diverged:')
    for col, rois in mismatches:
        print(f'   {col}: {rois}')
    display(cmp)
    raise AssertionError('baseline branch does not reproduce max_peaks_100_variant.ipynb\'s variant branch')

print(f"All {len(cmp)} ROIs' tm_score (max_peaks=100) branch matches "
      f"{MAX_PEAKS_ORACLE_TIMING_CSV}/{MAX_PEAKS_ORACLE_PRECISION_CSV} exactly on "
      f"seed_ann_id / base_size / n_detections / map_median / mad_scale / "
      f"tp_at_{{{', '.join(str(k) for k in BUDGETS_FOR_EVAL)}}} -- this notebook's harness "
      f"reproduces max_peaks_100_variant.ipynb's pipeline exactly before chromatin_od is "
      f"introduced.")

All 14 ROIs' tm_score (max_peaks=100) branch matches max_peaks_100_timing.csv/max_peaks_100_precision.csv exactly on seed_ann_id / base_size / n_detections / map_median / mad_scale / tp_at_{10, 20, 30} -- this notebook's harness reproduces max_peaks_100_variant.ipynb's pipeline exactly before chromatin_od is introduced.


## Sanity checks -- chromatin_od (od51) on the max_peaks=100 survivor pool

No notebook in this repo has previously ranked a `max_peaks`-capped, post-NMS pool by `chromatin_od`. The existing chromatin oracle (`results/precision_at_k_14roi_prodseed_chromatin_halfpixfix_raw.csv`) computed od51 on the *unbounded* pool (n_detections in the 15,000-19,000 range) -- a different candidate set from the ~90-99 survivors here -- so its `tp_at_budget` is not a valid cross-check for this combination, and this cell does not claim one. What can be checked directly: `od51` must be finite for every survivor in every ROI. `OD_PAD_51=25` was validated against a 60px pad on the *unbounded* pool (`latency_profiling/chromatin_od_latency_profile.ipynb`); this confirms that same minimal pad still loses no coverage on the much smaller capped pool.

In [6]:
print(TIMING[['n_detections', 'chromatin_od_nan_rate', 'chromatin_od_largest_tie_block']])
assert (TIMING['chromatin_od_nan_rate'] == 0.0).all(), (
    'od51 has NaN values on the max_peaks=100 pool -- OD_PAD_51=25 may be insufficient here')
print(f"\nod51 is defined for every candidate in every ROI (nan_rate=0.0 throughout); "
      f"largest_tie_block ranges {int(TIMING['chromatin_od_largest_tie_block'].min())}-"
      f"{int(TIMING['chromatin_od_largest_tie_block'].max())} out of "
      f"{int(TIMING['n_detections'].min())}-{int(TIMING['n_detections'].max())} candidates per ROI.")

           n_detections  chromatin_od_nan_rate  chromatin_od_largest_tie_block
file_name                                                                     
013.tiff             97                    0.0                               1
094.tiff             96                    0.0                               1
201.tiff             98                    0.0                               1
233.tiff             97                    0.0                               1
245.tiff             89                    0.0                               1
246.tiff             96                    0.0                               1
300.tiff             98                    0.0                               1
301.tiff             98                    0.0                               1
402.tiff             98                    0.0                               1
403.tiff             98                    0.0                               1
459.tiff             99                    0.0      

## Table 1 -- per-ROI stage timing, baseline vs. variant (ms)

In [7]:
STAGE_COLS_SHARED = ['t1_refine_seed_box_s', 't2_patch_template_build_s',
                    't3_template_matching_s', 't4_threshold_peak_extraction_s', 't5_nms_selfhit_s']
BRANCH_COLS = ['t6_baseline_s', 't6a_od_pad_s', 't6b_od51_loop_s', 't6c_variant_rank_s']
TOTAL_COLS = ['t_shared_stages_s', 't_chromatin_od_overhead_s',
             't_baseline_pipeline_s', 't_variant_pipeline_s', 't_outer_total_s']
ALL_TIME_COLS = ['t_setup_s'] + STAGE_COLS_SHARED + BRANCH_COLS + TOTAL_COLS

TIMING_MS = TIMING.copy()
for c in ALL_TIME_COLS:
    TIMING_MS[c[:-2] + '_ms'] = (TIMING_MS[c] * 1000).round(2)

display_cols = ['tumor_type', 'base_size', 'n_detections',
               't_setup_ms', 't1_refine_seed_box_ms', 't2_patch_template_build_ms',
               't3_template_matching_ms', 't4_threshold_peak_extraction_ms', 't5_nms_selfhit_ms',
               't6_baseline_ms', 't6a_od_pad_ms', 't6b_od51_loop_ms', 't6c_variant_rank_ms',
               't_baseline_pipeline_ms', 't_variant_pipeline_ms']
TIMING_MS.to_csv('chromatin_od_ranker_timing.csv')
print(f'-> chromatin_od_ranker_timing.csv')
TIMING_MS[display_cols]

-> chromatin_od_ranker_timing.csv


,tumor_type,base_size,n_detections,t_setup_ms,t1_refine_seed_box_ms,t2_patch_template_build_ms,t3_template_matching_ms,t4_threshold_peak_extraction_ms,t5_nms_selfhit_ms,t6_baseline_ms,t6a_od_pad_ms,t6b_od51_loop_ms,t6c_variant_rank_ms,t_baseline_pipeline_ms,t_variant_pipeline_ms
file_name,,,,,,,,,,,,,,,
013.tiff,human breast cancer,31,97,4154.77,1.03,0.03,1514.60,289.36,0.84,0.32,104.90,5.37,0.43,1806.17,1916.55
094.tiff,human breast cancer,25,96,3328.90,0.88,0.03,1037.53,299.28,0.71,0.23,33.41,3.94,0.34,1338.66,1376.11
201.tiff,canine lung cancer,51,98,2555.11,1.52,0.03,1210.62,256.42,0.72,0.23,34.55,5.93,0.59,1469.54,1510.37
233.tiff,canine lung cancer,25,97,2647.32,1.48,0.03,936.85,247.91,0.81,0.21,27.14,4.72,0.46,1187.29,1219.40
245.tiff,canine lymphosarcoma,47,89,2692.43,1.75,0.03,1100.76,242.52,0.67,0.21,25.16,5.95,0.46,1345.93,1377.29
246.tiff,canine lymphosarcoma,41,96,2459.00,1.87,0.02,1023.46,243.51,0.75,0.20,30.03,4.06,0.32,1269.82,1304.03
300.tiff,canine cutaneous mast cell tumor,45,98,2383.63,1.29,0.02,988.86,223.76,0.67,0.27,24.64,4.42,0.39,1214.88,1244.05
301.tiff,canine cutaneous mast cell tumor,41,98,2374.54,1.16,0.03,970.57,220.26,0.67,0.20,23.65,4.13,0.33,1192.88,1220.80
402.tiff,human neuroendocrine tumor,29,98,2963.65,0.92,0.02,1077.86,295.40,0.69,0.20,32.22,4.47,0.33,1375.10,1411.92


In [8]:
print('Shared stages 1-5 (identical between branches by construction -- one matchTemplate pass, '
     'one extraction, one NMS; MAX_PEAKS=100 is fixed and identical in both branches here, unlike '
     'max_peaks_100_variant.ipynb where the cap itself changed this pool between branches):')
for col in ['t1_refine_seed_box_ms', 't2_patch_template_build_ms', 't3_template_matching_ms',
           't4_threshold_peak_extraction_ms', 't5_nms_selfhit_ms']:
    print(f'  {col:34s} mean={TIMING_MS[col].mean():7.2f}ms')

print()
print('Stage 6, baseline (tm_score) vs. variant (chromatin_od) -- the ablated step:')
print(f"  baseline: sort by tm_score              {TIMING_MS['t6_baseline_ms'].mean():8.3f}ms mean")
print(f"  variant:  pad (t6a)                      {TIMING_MS['t6a_od_pad_ms'].mean():8.3f}ms mean")
print(f"  variant:  od51 feature loop (t6b)        {TIMING_MS['t6b_od51_loop_ms'].mean():8.3f}ms mean")
print(f"  variant:  sort by chromatin_od (t6c)     {TIMING_MS['t6c_variant_rank_ms'].mean():8.3f}ms mean")
od_overhead_mean_ms = TIMING_MS['t_chromatin_od_overhead_ms'].mean()
tm_stage6_mean_ms = TIMING_MS['t6_baseline_ms'].mean()
print(f"  variant stage-6 total (t6a+t6b+t6c)      {od_overhead_mean_ms:8.3f}ms mean "
     f"({od_overhead_mean_ms / tm_stage6_mean_ms:.1f}x the baseline's stage-6 cost)")

print()
d_pipeline_ms = TIMING_MS['t_variant_pipeline_ms'].mean() - TIMING_MS['t_baseline_pipeline_ms'].mean()
print(f"Full pipeline (stages 1-6) mean: "
     f"baseline={TIMING_MS['t_baseline_pipeline_ms'].mean():.2f}ms  "
     f"variant={TIMING_MS['t_variant_pipeline_ms'].mean():.2f}ms  delta={d_pipeline_ms:+.2f}ms "
     f"({d_pipeline_ms / TIMING_MS['t_baseline_pipeline_ms'].mean() * 100:+.1f}%)")

Shared stages 1-5 (identical between branches by construction -- one matchTemplate pass, one extraction, one NMS; MAX_PEAKS=100 is fixed and identical in both branches here, unlike max_peaks_100_variant.ipynb where the cap itself changed this pool between branches):
  t1_refine_seed_box_ms              mean=   1.26ms
  t2_patch_template_build_ms         mean=   0.02ms
  t3_template_matching_ms            mean=1110.41ms
  t4_threshold_peak_extraction_ms    mean= 258.68ms
  t5_nms_selfhit_ms                  mean=   0.71ms

Stage 6, baseline (tm_score) vs. variant (chromatin_od) -- the ablated step:
  baseline: sort by tm_score                 0.219ms mean
  variant:  pad (t6a)                        40.095ms mean
  variant:  od51 feature loop (t6b)           4.619ms mean
  variant:  sort by chromatin_od (t6c)        0.381ms mean
  variant stage-6 total (t6a+t6b+t6c)        45.094ms mean (205.6x the baseline's stage-6 cost)

Full pipeline (stages 1-6) mean: baseline=1371.30ms  variant=14

## Table 2 -- precision@{10,20,30}, baseline vs. variant

Long format: one row per (ROI, branch, budget). `budget_delivered` is reported explicitly rather than assumed to equal K. `branch` maps `arm='tm_score' -> 'baseline'`, `arm='chromatin_od' -> 'variant'`, so Tables 2/2b and the Summary below read identically to `max_peaks_100_variant.ipynb`'s.

In [9]:
ALL_RAW['branch'] = ALL_RAW['arm'].map({'tm_score': 'baseline', 'chromatin_od': 'variant'})
ALL_RAW['precision_at_budget'] = (ALL_RAW['tp_at_budget']
                                  / ALL_RAW['budget_delivered'].replace(0, np.nan))

PRECISION_LONG = ALL_RAW[['file_name', 'tumor_type', 'branch', 'arm', 'budget', 'n_detections',
                          'n_gt_mitotic', 'budget_delivered', 'tp_at_budget',
                          'precision_at_budget', 'recall_at_budget']].copy()
PRECISION_LONG = PRECISION_LONG.sort_values(['file_name', 'budget', 'branch']).reset_index(drop=True)
PRECISION_LONG.to_csv('chromatin_od_ranker_precision.csv', index=False)
print(f'-> chromatin_od_ranker_precision.csv  ({len(PRECISION_LONG)} rows = 14 ROIs x 2 branches x '
     f'{len(BUDGETS_FOR_EVAL)} budgets)')

for branch in ['baseline', 'variant']:
    sub = PRECISION_LONG[(PRECISION_LONG['branch'] == branch) & (PRECISION_LONG['budget'] == BUDGET)]
    n_starved = int((sub['budget_delivered'] < BUDGET).sum())
    print(f'{branch:10s}: delivers fewer than BUDGET={BUDGET} candidates on '
         f'{n_starved}/{len(sub)} ROIs at the largest reported budget')

PRECISION_LONG.round(4)

-> chromatin_od_ranker_precision.csv  (84 rows = 14 ROIs x 2 branches x 3 budgets)
baseline  : delivers fewer than BUDGET=30 candidates on 0/14 ROIs at the largest reported budget
variant   : delivers fewer than BUDGET=30 candidates on 0/14 ROIs at the largest reported budget


,file_name,tumor_type,branch,arm,budget,n_detections,n_gt_mitotic,budget_delivered,tp_at_budget,precision_at_budget,recall_at_budget
0,013.tiff,human breast cancer,baseline,tm_score,10,97,17,10,3,0.3000,0.1765
1,013.tiff,human breast cancer,variant,chromatin_od,10,97,17,10,6,0.6000,0.3529
2,013.tiff,human breast cancer,baseline,tm_score,20,97,17,20,4,0.2000,0.2353
3,013.tiff,human breast cancer,variant,chromatin_od,20,97,17,20,9,0.4500,0.5294
4,013.tiff,human breast cancer,baseline,tm_score,30,97,17,30,6,0.2000,0.3529
5,013.tiff,human breast cancer,variant,chromatin_od,30,97,17,30,9,0.3000,0.5294
6,094.tiff,human breast cancer,baseline,tm_score,10,96,81,10,4,0.4000,0.0494
7,094.tiff,human breast cancer,variant,chromatin_od,10,96,81,10,5,0.5000,0.0617
8,094.tiff,human breast cancer,baseline,tm_score,20,96,81,20,10,0.5000,0.1235
9,094.tiff,human breast cancer,variant,chromatin_od,20,96,81,20,15,0.7500,0.1852


## Table 2b -- precision@K pivoted for readability

In [10]:
PRECISION_PIVOT = PRECISION_LONG.pivot_table(
    index=['tumor_type', 'file_name'], columns=['budget', 'branch'], values='precision_at_budget'
)
PRECISION_PIVOT.round(4)

budget                                           10               20               30        
branch                                     baseline variant baseline variant baseline variant
tumor_type                       file_name                                                   
canine cutaneous mast cell tumor 300.tiff       0.7     0.9     0.70    0.90   0.6667  0.8667
                                 301.tiff       0.9     0.8     0.90    0.90   0.8000  0.9000
canine lung cancer               201.tiff       0.3     0.4     0.25    0.40   0.2000  0.3000
                                 233.tiff       0.3     0.7     0.25    0.40   0.2667  0.3333
canine lymphosarcoma             245.tiff       0.1     0.2     0.10    0.10   0.1000  0.0667
                                 246.tiff       1.0     1.0     0.90    0.70   0.8000  0.7000
canine soft tissue sarcoma       459.tiff       0.6     0.9     0.55    0.85   0.6333  0.8333
                                 460.tiff       0.7     0.7     0.50    0.80   0.4000  0.5667
human breast cancer              013.tiff       0.3     0.6     0.20    0.45   0.2000  0.3000
                                 094.tiff       0.4     0.5     0.50    0.75   0.5333  0.7333
human melanoma                   529.tiff       0.3     0.6     0.20    0.35   0.1333  0.2333
                                 548.tiff       0.5     0.8     0.50    0.65   0.5333  0.7000
human neuroendocrine tumor       402.tiff       0.4     0.7     0.45    0.60   0.4667  0.5000
                                 403.tiff       0.5     0.5     0.35    0.50   0.2667  0.4667

## Summary -- pooled precision and win counts, by budget

In [11]:
print(f'Pooled precision (sum tp / sum delivered) across all 14 ROIs, by branch and budget:')
for k in BUDGETS_FOR_EVAL:
    print(f'  K={k}:')
    for branch in ['baseline', 'variant']:
        sub = PRECISION_LONG[(PRECISION_LONG['branch'] == branch) & (PRECISION_LONG['budget'] == k)]
        pooled = sub['tp_at_budget'].sum() / sub['budget_delivered'].sum()
        print(f'    {branch:10s}: {pooled:.4f}  ({int(sub["tp_at_budget"].sum())} tp / '
             f'{int(sub["budget_delivered"].sum())} delivered)')

print()
for k in BUDGETS_FOR_EVAL:
    base_k = PRECISION_LONG[(PRECISION_LONG['branch'] == 'baseline') & (PRECISION_LONG['budget'] == k)].set_index('file_name')
    var_k = PRECISION_LONG[(PRECISION_LONG['branch'] == 'variant') & (PRECISION_LONG['budget'] == k)].set_index('file_name')
    d_var = (var_k['precision_at_budget'] - base_k['precision_at_budget'])
    r_var = (var_k['recall_at_budget'] < base_k['recall_at_budget']).sum()
    print(f'K={k}: variant precision delta: {int((d_var > 0).sum())} up / '
         f'{int((d_var < 0).sum())} down / {int((d_var == 0).sum())} unchanged of 14 ROIs; '
         f'recall lower on {int(r_var)}/14')

print()
print('Note: precision_at_budget = tp/budget_delivered and recall_at_budget = tp/n_gt_mitotic '
     'share the same numerator (tp) with fixed denominators here (budget_delivered=K on every '
     'ROI at these budgets, n_gt_mitotic fixed per ROI) -- "recall lower" above is the same ROIs '
     'as "down" in precision, not an independent tradeoff.')

Pooled precision (sum tp / sum delivered) across all 14 ROIs, by branch and budget:
  K=10:
    baseline  : 0.5000  (70 tp / 140 delivered)
    variant   : 0.6643  (93 tp / 140 delivered)
  K=20:
    baseline  : 0.4536  (127 tp / 280 delivered)
    variant   : 0.5964  (167 tp / 280 delivered)
  K=30:
    baseline  : 0.4286  (180 tp / 420 delivered)
    variant   : 0.5357  (225 tp / 420 delivered)

K=10: variant precision delta: 10 up / 1 down / 3 unchanged of 14 ROIs; recall lower on 1/14
K=20: variant precision delta: 11 up / 1 down / 2 unchanged of 14 ROIs; recall lower on 1/14
K=30: variant precision delta: 12 up / 2 down / 0 unchanged of 14 ROIs; recall lower on 2/14

Note: precision_at_budget = tp/budget_delivered and recall_at_budget = tp/n_gt_mitotic share the same numerator (tp) with fixed denominators here (budget_delivered=K on every ROI at these budgets, n_gt_mitotic fixed per ROI) -- "recall lower" above is the same ROIs as "down" in precision, not an independent tradeoff.


## Closing readout

In [12]:
print(f'{VARIANT_LABEL} vs. baseline (tm_score, max_peaks=100), mean over {len(TIMING)} ROIs:')
print(f'  full pipeline (stages 1-6): {TIMING_MS["t_baseline_pipeline_ms"].mean():.2f}ms -> '
     f'{TIMING_MS["t_variant_pipeline_ms"].mean():.2f}ms ({d_pipeline_ms:+.2f}ms, '
     f'{d_pipeline_ms / TIMING_MS["t_baseline_pipeline_ms"].mean() * 100:+.1f}%)')
print()
print('Precision, pooled across 14 ROIs (baseline -> variant):')
for k in BUDGETS_FOR_EVAL:
    b = PRECISION_LONG[(PRECISION_LONG['branch'] == 'baseline') & (PRECISION_LONG['budget'] == k)]
    v = PRECISION_LONG[(PRECISION_LONG['branch'] == 'variant') & (PRECISION_LONG['budget'] == k)]
    pb = b['tp_at_budget'].sum() / b['budget_delivered'].sum()
    pv = v['tp_at_budget'].sum() / v['budget_delivered'].sum()
    n_starved = int((v['budget_delivered'] < k).sum())
    print(f'  K={k}: {pb:.4f} -> {pv:.4f}  (under-delivers K={k} on {n_starved}/14 ROIs)')
print()
print(f'Caveat (D5, restated): this is single-seed, n=14 -- one click per ROI, not the paired, '
     f'multi-seed sweep D5 itself sets as the bar for changing a production default. The '
     f'baseline (tm_score) precision numbers above reproduce max_peaks_100_variant.ipynb\'s '
     f'already-committed variant-branch numbers (Verification 1); the chromatin_od numbers are '
     f'new -- no notebook in this repo has measured this exact combination (max_peaks=100 cap, '
     f'then re-rank by chromatin_od) before.')
print(f'Whether this is specifically a "cap + rerank" cascade effect, or whether chromatin_od '
     f'simply beats tm_score at these budgets/seeds with or without the cap, is checked in the '
     f'final cell below against the already-committed uncapped oracle.')
print(f'\nnotebook ran in {time.time() - NB_T0:.0f}s')

chromatin_od (od51) ranking, max_peaks=100 vs. baseline (tm_score, max_peaks=100), mean over 14 ROIs:
  full pipeline (stages 1-6): 1371.30ms -> 1416.17ms (+44.87ms, +3.3%)

Precision, pooled across 14 ROIs (baseline -> variant):
  K=10: 0.5000 -> 0.6643  (under-delivers K=10 on 0/14 ROIs)
  K=20: 0.4536 -> 0.5964  (under-delivers K=20 on 0/14 ROIs)
  K=30: 0.4286 -> 0.5357  (under-delivers K=30 on 0/14 ROIs)

Caveat (D5, restated): this is single-seed, n=14 -- one click per ROI, not the paired, multi-seed sweep D5 itself sets as the bar for changing a production default. The baseline (tm_score) precision numbers above reproduce max_peaks_100_variant.ipynb's already-committed variant-branch numbers (Verification 1); the chromatin_od numbers are new -- no notebook in this repo has measured this exact combination (max_peaks=100 cap, then re-rank by chromatin_od) before.
Whether this is specifically a "cap + rerank" cascade effect, or whether chromatin_od simply beats tm_score at these bu

## Added-step accounting -- time added by the new ranking criterion

What the task asked for explicitly, isolated from the tables above: precisely how much wall-clock the `chromatin_od` re-rank adds on top of the existing `tm_score`-ranked, `max_peaks=100`-capped pipeline, not netted against the sort it replaces (`t6_baseline`, already small at this pool size).

In [13]:
od_overhead_s = TIMING['t_chromatin_od_overhead_s']
pad_s = TIMING['t6a_od_pad_s']
new_work_s = TIMING['t6b_od51_loop_s'] + TIMING['t6c_variant_rank_s']  # loop + resort only

print('=== Time added by switching the ranker to chromatin_od (max_peaks=100 already applied) ===')
print(f'Per-ROI cost of stages 6a-6c on the '
     f'{int(TIMING["n_detections"].min())}-{int(TIMING["n_detections"].max())}-candidate capped '
     f'pool, n=14 ROIs:')
print(f'  mean    = {od_overhead_s.mean()*1000:7.3f}ms')
print(f'  median  = {od_overhead_s.median()*1000:7.3f}ms')
print(f'  std     = {od_overhead_s.std()*1000:7.3f}ms')
print(f'  min/max = {od_overhead_s.min()*1000:7.3f}ms / {od_overhead_s.max()*1000:7.3f}ms')
print(f'  total over all 14 ROIs = {od_overhead_s.sum()*1000:7.2f}ms ({od_overhead_s.sum():.3f}s)')
print()

outliers = od_overhead_s.sort_values(ascending=False).head(2)
others_mean_ms = od_overhead_s.drop(outliers.index).mean() * 1000
print(f'Two ROIs run well above the rest: '
     + ', '.join(f'{fn} ({v*1000:.1f}ms)' for fn, v in outliers.items())
     + f', vs. {others_mean_ms:.1f}ms mean on the other 12 -- these two pull the mean well above '
       f'the median.')
print()

# cv2.copyMakeBorder pads the whole ROI array, so its cost should scale with ROI pixel count,
# not with pool size. Check that directly rather than asserting it.
roi_px = TIMING['roi_pixels']
pad_ms = TIMING['t6a_od_pad_s'] * 1000
big_px = roi_px == roi_px.max()
print(f'cv2.copyMakeBorder cost vs. ROI pixel count (corr={pad_ms.corr(roi_px):.2f}): the '
     f'{int(big_px.sum())} largest-pixel ROIs ({roi_px[big_px].iloc[0]:,} px) pad at '
     f'{pad_ms[big_px].median():.1f}ms median vs. {pad_ms[~big_px].median():.1f}ms median for '
     f'the other {int((~big_px).sum())} -- so pixel count does explain a real baseline effect, '
     f'contrary to an earlier pass over this notebook that checked only whether same-sized ROIs '
     f'spike identically (they need not, if a second effect is layered on top) and concluded '
     f'wrongly that dimensions explain nothing.')
outlier_in_big = [fn for fn in outliers.index if big_px.get(fn, False)]
if outlier_in_big:
    same_class_others = pad_ms[big_px].drop(outlier_in_big)
    print(f'{", ".join(outlier_in_big)} still run well above their OWN size class '
         f'({pad_ms[outlier_in_big].mean():.1f}ms vs. {same_class_others.mean():.1f}ms mean for '
         f'the other largest-pixel ROIs) -- so a second, per-call effect sits on top of the area '
         f'baseline. This run cannot separate a stable per-ROI cause from run-to-run jitter in a '
         f'single `copyMakeBorder` call (that needs repeated timing, not done here), so which of '
         f'the two it is stays unattributed -- but "dimensions explain nothing" is not right.')
print()

print('Decomposed -- the honest answer to "how much time is added by the new criteria":')
print(f'  t6a (cv2.copyMakeBorder on the full ROI, independent of pool size):     '
     f'mean={pad_s.mean()*1000:7.3f}ms  median={pad_s.median()*1000:6.3f}ms  '
     f'({pad_s.mean() / od_overhead_s.mean() * 100:.0f}% of the added time)')
print(f'  t6b+t6c (od51 loop + re-sort -- the part that actually scales with pool size): '
     f'mean={new_work_s.mean()*1000:7.3f}ms  median={new_work_s.median()*1000:6.3f}ms  '
     f'({new_work_s.mean() / od_overhead_s.mean() * 100:.0f}% of the added time)')
print()
print('The pad dominates here precisely because it is fixed-cost: `cv2.copyMakeBorder` pads the '
     'whole ROI array regardless of how many candidates get ranked afterward, so shrinking the '
     'pool from ~17,000 to ~97 candidates cannot shrink it. Only t6b+t6c scale with pool size, '
     'and at this pool size they cost next to nothing.')
print('This pad is also an implementation choice, not an intrinsic cost of chromatin_od: it '
     'exists only so a border-adjacent candidate can still read a full 51px window, and a '
     'per-candidate border check would serve the same purpose on a ~97-candidate pool without '
     'padding the whole ROI. The criterion itself -- compute od51, re-sort -- is the ~4.8ms '
     'figure (t6b+t6c), not the ~45ms combined with the pad; that alternative implementation is '
     'not measured here, so this notebook reports the pad as-is rather than a hypothetical.')
print()

d_pipeline_ms = TIMING_MS['t_variant_pipeline_ms'].mean() - TIMING_MS['t_baseline_pipeline_ms'].mean()
print(f"Full-pipeline delta, baseline -> variant: "
     f"{TIMING_MS['t_baseline_pipeline_ms'].mean():.2f}ms -> "
     f"{TIMING_MS['t_variant_pipeline_ms'].mean():.2f}ms "
     f"({d_pipeline_ms:+.2f}ms, "
     f"{d_pipeline_ms / TIMING_MS['t_baseline_pipeline_ms'].mean() * 100:+.1f}%).")
print()
try:
    unbounded = pd.read_csv('../latency_profiling/chromatin_od_latency_per_roi.csv')
    unbounded_pad_ms = unbounded['t7a_od_pad_s'].mean() * 1000
    unbounded_loop_ms = unbounded['t7b_od51_loop_s'].mean() * 1000
    unbounded_n_det = unbounded['n_detections'].mean()
    n_det_mean = TIMING['n_detections'].mean()
    print(f'For context: latency_profiling/chromatin_od_latency_profile.ipynb ran the same pad '
         f'operation on the same-sized ROIs but an *unbounded* pool '
         f'(n_detections~{unbounded_n_det:,.0f}, {unbounded_n_det / n_det_mean:.0f}x more '
         f'candidates ranked afterward). Its pad cost there was {unbounded_pad_ms:.1f}ms mean -- '
         f'close to this notebook\'s {pad_s.mean()*1000:.1f}ms, as expected for a pool-size-'
         f'independent operation. Its od51-loop cost there was {unbounded_loop_ms:.1f}ms mean, '
         f'against {TIMING_MS["t6b_od51_loop_ms"].mean():.2f}ms here -- that is the part that '
         f'actually collapses with the smaller pool.')
except FileNotFoundError:
    print('(latency_profiling/chromatin_od_latency_per_roi.csv not found -- skipping the '
         'unbounded-pool comparison.)')

=== Time added by switching the ranker to chromatin_od (max_peaks=100 already applied) ===
Per-ROI cost of stages 6a-6c on the 89-99-candidate capped pool, n=14 ROIs:
  mean    =  45.094ms
  median  =  35.715ms
  std     =  28.453ms
  min/max =  28.120ms / 112.620ms
  total over all 14 ROIs =  631.31ms (0.631s)

Two ROIs run well above the rest: 403.tiff (112.6ms), 013.tiff (110.7ms), vs. 34.0ms mean on the other 12 -- these two pull the mean well above the median.

cv2.copyMakeBorder cost vs. ROI pixel count (corr=0.56): the 5 largest-pixel ROIs (39,047,580 px) pad at 33.0ms median vs. 26.2ms median for the other 9 -- so pixel count does explain a real baseline effect, contrary to an earlier pass over this notebook that checked only whether same-sized ROIs spike identically (they need not, if a second effect is layered on top) and concluded wrongly that dimensions explain nothing.
403.tiff, 013.tiff still run well above their OWN size class (106.4ms vs. 32.6ms mean for the other large

## Precision boost from the added ranking step

The other half of the task's ask: whatever `chromatin_od` buys in precision@K once `max_peaks=100` has already capped the field, stated directly rather than left to be read off Table 2b.

In [14]:
print('=== Precision boost from ranking by chromatin_od instead of tm_score (max_peaks=100) ===')
pooled = {}
for k in BUDGETS_FOR_EVAL:
    b = PRECISION_LONG[(PRECISION_LONG['branch'] == 'baseline') & (PRECISION_LONG['budget'] == k)]
    v = PRECISION_LONG[(PRECISION_LONG['branch'] == 'variant') & (PRECISION_LONG['budget'] == k)]
    pb = b['tp_at_budget'].sum() / b['budget_delivered'].sum()
    pv = v['tp_at_budget'].sum() / v['budget_delivered'].sum()
    d = (v.set_index('file_name')['precision_at_budget']
        - b.set_index('file_name')['precision_at_budget'])
    pooled[k] = dict(pb=pb, pv=pv, delta=pv - pb, up=int((d > 0).sum()),
                     down=int((d < 0).sum()), unchanged=int((d == 0).sum()))
    pct = (pv - pb) / pb * 100 if pb else float('nan')
    print(f'  K={k:2d}: pooled precision {pb:.4f} -> {pv:.4f}  '
         f'(delta {pv - pb:+.4f}, {pct:+.1f}%)  |  '
         f"per-ROI: {pooled[k]['up']} up / {pooled[k]['down']} down / "
         f"{pooled[k]['unchanged']} unchanged of 14")

print()
delta_str = ', '.join(f"K={k}: {pooled[k]['delta']:+.4f}" for k in BUDGETS_FOR_EVAL)
new_work_ms = (TIMING['t6b_od51_loop_s'] + TIMING['t6c_variant_rank_s']).mean() * 1000
pad_ms = TIMING['t6a_od_pad_s'].mean() * 1000
print(f'Bottom line: the criterion itself -- computing od51 and re-sorting on this '
     f'~97-candidate pool -- costs ~{new_work_ms:.1f}ms/ROI. This implementation also pays a '
     f'~{pad_ms:.1f}ms `cv2.copyMakeBorder` pad on the whole ROI regardless of pool size (an '
     f'implementation choice, not an intrinsic cost of chromatin_od -- a per-candidate border '
     f'check was not built or measured here), so the combined stage-6 delta actually observed '
     f'is ~{od_overhead_s.mean()*1000:.1f}ms/ROI '
     f'(+{od_overhead_s.mean() / TIMING["t_baseline_pipeline_s"].mean() * 100:.1f}% of the full '
     f'click-to-list pipeline). These ms figures are single-run point estimates -- expect them '
     f'to shift somewhat on a re-run -- though the pad/criterion split is expected to stay '
     f'lopsided on physical grounds: the pad is O(ROI pixels) and the loop is O(pool size), so '
     f'at a ~97-candidate pool the pad dominates regardless of the exact ms. Against that cost: '
     f'a pooled precision@K change of {delta_str}; these tp_at_budget values were independently '
     f're-derived from raw pixels in an audit of this notebook (18/18 matched, 0 divergences), '
     f'which is stronger evidence of correctness than re-running this notebook a second time '
     f'would be.')
print()
print('The baseline (tm_score) numbers reproduce max_peaks_100_variant.ipynb\'s already-committed '
     'variant branch (Verification 1); the chromatin_od numbers here are new measurements. This '
     'run is single-seed/n=14 -- treat this as a cost-and-effect measurement at one operating '
     'point, not a decision on whether to switch rankers. Whether the precision gain is specific '
     'to this capped operating point or holds more generally is checked in the next cell.')

=== Precision boost from ranking by chromatin_od instead of tm_score (max_peaks=100) ===
  K=10: pooled precision 0.5000 -> 0.6643  (delta +0.1643, +32.9%)  |  per-ROI: 10 up / 1 down / 3 unchanged of 14
  K=20: pooled precision 0.4536 -> 0.5964  (delta +0.1429, +31.5%)  |  per-ROI: 11 up / 1 down / 2 unchanged of 14
  K=30: pooled precision 0.4286 -> 0.5357  (delta +0.1071, +25.0%)  |  per-ROI: 12 up / 2 down / 0 unchanged of 14

Bottom line: the criterion itself -- computing od51 and re-sorting on this ~97-candidate pool -- costs ~5.0ms/ROI. This implementation also pays a ~40.1ms `cv2.copyMakeBorder` pad on the whole ROI regardless of pool size (an implementation choice, not an intrinsic cost of chromatin_od -- a per-candidate border check was not built or measured here), so the combined stage-6 delta actually observed is ~45.1ms/ROI (+3.3% of the full click-to-list pipeline). These ms figures are single-run point estimates -- expect them to shift somewhat on a re-run -- though the 

## Does max_peaks=100 create this advantage, or just come along for it?

An audit of this notebook flagged the earlier "two-stage cascade" framing as unverified: it asserted, without checking, that the precision gain depends on the cap having pre-filtered the pool by tm_score. The uncapped pipeline on these exact same seeds is already committed -- `results/precision_at_k_14roi_prodseed_chromatin_halfpixfix_raw.csv` differs from this notebook's config only in `MAX_PEAKS=2,000,000` vs `100` (same seed_ann_id, base_size, map_median, mad_scale, n_gt_mitotic on all 14 ROIs) -- so this checks it directly instead of asserting it.

In [15]:
uncapped = pd.read_csv('../results/precision_at_k_14roi_prodseed_chromatin_halfpixfix_raw.csv')
uncapped_tm = uncapped[uncapped['arm'] == 'tm_score']
uncapped_od = uncapped[uncapped['arm'] == 'chromatin_od']

print('Pooled precision@K -- capped (this notebook) vs. uncapped (already-committed oracle), '
     'same 14 ROIs/seeds:')
cap_contribution = {}
for k in BUDGETS_FOR_EVAL:
    utm = uncapped_tm[uncapped_tm['budget'] == k]
    uod = uncapped_od[uncapped_od['budget'] == k]
    p_tm_uncapped = utm['tp_at_budget'].sum() / utm['budget_delivered'].sum()
    p_od_uncapped = uod['tp_at_budget'].sum() / uod['budget_delivered'].sum()
    uncapped_gain = p_od_uncapped - p_tm_uncapped
    capped_gain = pooled[k]['delta']
    cap_contribution[k] = capped_gain - uncapped_gain
    print(f'  K={k:2d}: uncapped tm_score={p_tm_uncapped:.4f} (capped was '
         f'{pooled[k]["pb"]:.4f} -- {"identical" if abs(p_tm_uncapped - pooled[k]["pb"]) < 1e-9 else "DIFFERENT, see note below"}), '
         f'uncapped chromatin_od={p_od_uncapped:.4f} (capped was {pooled[k]["pv"]:.4f})  |  '
         f'uncapped gain {uncapped_gain:+.4f}  vs.  capped gain {capped_gain:+.4f}  |  '
         f'max_peaks=100\'s own marginal contribution: {cap_contribution[k]:+.4f}')

print()
print('tm_score\'s pooled precision is identical capped and uncapped at every K -- expected, '
     'since max_peaks=100 keeps the top-100 candidates *by score* pre-NMS, so nothing that '
     'could rank in a top-30-by-score list post-NMS is ever excluded by that cap. Only the '
     'chromatin_od row moves, so `cap_contribution` above is exactly '
     '(capped chromatin_od precision) - (uncapped chromatin_od precision).')
print()
print('So: chromatin_od beats tm_score at these budgets and seeds WITH or WITHOUT the '
     'max_peaks=100 cap -- this is not a cascade effect the cap creates. The cap\'s own '
     'marginal contribution is small and positive at K=10, and net NEGATIVE at K=20 and K=30: '
     'capping to 100 candidates before reranking by chromatin_od loses more precision than it '
     'adds at those budgets, relative to reranking the full uncapped pool.')
print()
print('This also means the earlier "does not contradict D5\'s unbounded-pool null" framing was '
     'not a real reconciliation: D5\'s null (midog_utils/chromatin.py\'s module docstring) is '
     'specifically Delta(recall@250) on a paired, 7-ROI x 5-seed sweep at z=1.0 -- a different '
     'metric, operating point and sample entirely from precision@10-30 on n=14 single-seed '
     'here. This result is outside D5\'s scope rather than reconciled with it: it neither '
     'confirms nor contradicts that finding, and does not on its own meet D5\'s bar for '
     'changing what the production ranker is.')

Pooled precision@K -- capped (this notebook) vs. uncapped (already-committed oracle), same 14 ROIs/seeds:
  K=10: uncapped tm_score=0.5000 (capped was 0.5000 -- identical), uncapped chromatin_od=0.6214 (capped was 0.6643)  |  uncapped gain +0.1214  vs.  capped gain +0.1643  |  max_peaks=100's own marginal contribution: +0.0429


  K=20: uncapped tm_score=0.4536 (capped was 0.4536 -- identical), uncapped chromatin_od=0.6107 (capped was 0.5964)  |  uncapped gain +0.1571  vs.  capped gain +0.1429  |  max_peaks=100's own marginal contribution: -0.0143
  K=30: uncapped tm_score=0.4286 (capped was 0.4286 -- identical), uncapped chromatin_od=0.5595 (capped was 0.5357)  |  uncapped gain +0.1310  vs.  capped gain +0.1071  |  max_peaks=100's own marginal contribution: -0.0238

tm_score's pooled precision is identical capped and uncapped at every K -- expected, since max_peaks=100 keeps the top-100 candidates *by score* pre-NMS, so nothing that could rank in a top-30-by-score list post-NMS is ever excluded by that cap. Only the chromatin_od row moves, so `cap_contribution` above is exactly (capped chromatin_od precision) - (uncapped chromatin_od precision).

So: chromatin_od beats tm_score at these budgets and seeds WITH or WITHOUT the max_peaks=100 cap -- this is not a cascade effect the cap creates. The cap's own margi